In [1]:
¿Qué tipo de parámetros admite la solicitud?
¿Qué tipo respuestas se pueden obtener con cada solicitud?
¿Qué tipo de restricciones tiene la API?

Object `solicitud` not found.
Object `solicitud` not found.
Object `API` not found.


## ¿Qué tipo de parámetros admite la solicitud?

De acuerdo a la documentación, la API acepta los siguientes parámetros:

| Parámetro        | Tipo       | Valor por defecto | Descripción |
|-----------------|------------|------------------|-------------|
| `api_key`        | string     | `DEMO_KEY`       | API key.|
| `date`           | YYYY-MM-DD | *hoy*            | Fecha de la imagen APOD. Debe de estar entre `1995-06-16` y la fecha actual. |
| `concept_tags`   | bool       | False            | Indica si deben devolverse etiquetas de concepto junto con la respuesta. |
| `hd`             | bool       | ignorado         | Indica si deben devolverse imágenes en alta resolución.|
| `count`          | int        | ninguno          | Entero positivo no mayor a `100`. Si se especifica, se devolverán `count` imágenes aleatorias en un arreglo JSON. No puede usarse junto con `date` ni con `start_date` y `end_date`. |
| `start_date`     | YYYY-MM-DD | ninguno          | Indica el inicio de un rango de fechas. Todas las imágenes desde `start_date` hasta `end_date` se devolverán en un arreglo JSON. No puede usarse con `date`. |
| `end_date`       | YYYY-MM-DD | *hoy*            | Indica el final de un rango de fechas. Si se especifica `start_date` sin `end_date`, este valor será la fecha actual. |
| `thumbs`         | bool       | False            | Indica si la API debe devolver una URL de miniatura para archivos de video. Si el APOD no es un video, este parámetro se ignora. |

## ¿Qué tipo respuestas se pueden obtener con cada solicitud?

| Campo devuelto     | Tipo        | Descripción |
|-------------------|------------|-------------|
| `resource`         | dictionary | Un diccionario que describe el `image_set` o planeta que ilustra la respuesta, determinado completamente por el endpoint estructurado. |
| `concept_tags`     | bool       | Reflejo booleano de la opción proporcionada. Se incluye en la respuesta debido a los valores predeterminados. |
| `title`            | string     | El título de la imagen. |
| `date`             | YYYY-MM-DD | Fecha de la imagen. Se incluye en la respuesta debido a los valores predeterminados. |
| `url`              | string (URL) | La URL de la imagen APOD o del video del día. |
| `hdurl`            | string (URL) | La URL de cualquier imagen en alta resolución para ese día. Se devuelve independientemente del parámetro `hd`, pero se omitirá si originalmente no existe en APOD. |
| `media_type`       | string     | El tipo de medio (dato) devuelto. Puede ser `image` o `video`, dependiendo del contenido. |
| `explanation`      | string     | El texto explicativo proporcionado sobre la imagen. |
| `concepts`         | array      | Los conceptos más relevantes dentro del texto explicativo. Solo se incluye si `concept_tags` está configurado como `True`. |
| `thumbnail_url`    | string (URL) | La URL de la miniatura del video. |
| `copyright`        | string     | El nombre del titular de los derechos de autor. |
| `service_version`  | string     | La versión del servicio utilizada. |

In [2]:
import requests
import pandas as pd
from time import sleep
from datetime import date

In [3]:
hoy = date.today().strftime("%Y-%m-%d")
hoy

'2026-05-17'

# Obteniendo data
De acuerdo a la documentación, la primera imagen es de 1995-06-16.

Para no agotar a la API solicitamos la data año por año. 
- Para el primer año (1995) solicitamos la data de 1995-06-16 a 1995-12-31. 
- Posteriormente iteramos sobre los años 1996 a 2025 solicitando la data de todo el año: año-01-01 a año-12-31
- Finalmente solicitamos la data de 2026 (2026-01-01 a la fecha actual)

In [4]:
# Definimos la API key y url para solicitar data
API_KEY = "mNs15zaI1BwwQguRRaQZRs460g7ZhlHA0jcUvszc"  # replace with your NASA API key
url = "https://api.nasa.gov/planetary/apod"

Para no agotar a la API solicitamos la data año por año. 
Hay dos casos especiales: El primer año (1995) y el año actual

In [5]:
params = {
    "api_key": API_KEY,
    "start_date": "1995-06-16",  # optional: YYYY-MM-DD
    "end_date": "1995-12-31",  # optional: YYYY-MM-DD
    }

# Se crea la lista response_data que gaurda la data en formato JSON 
response = requests.get(url, params=params, timeout=180)
response_data= response.json()

In [6]:
response.status_code

200

In [7]:
# Para no agotar a la API solicitamos la data año por año. 
# Hay dos casos especiales: El primer año (1995) y el año actual

# De acuerdo a la documentación de la API, la fecha más antigua es 1995-06-16
params = {
    "api_key": API_KEY,
    "start_date": "1995-06-16",  # optional: YYYY-MM-DD
    "end_date": "1995-12-31",  # optional: YYYY-MM-DD
    }

# Se crea la lista response_data que gaurda la data en formato JSON 
response = requests.get(url, params=params, timeout=180)
response_data= response.json()


# Se solicita la info para los años 1996 - 2025 cambiando los parámetros start_date y end_date
for anno in range(1996, 2026 ):

    success= False

    while success== False: 
        
        print(anno)
        # Cambiamos los parámetros al primer y último dia del anno
        params["start_date"]= f"{anno}-01-01"
        params["end_date"]= f"{anno}-12-31"
        
        sleep(10)
        response = requests.get(url, params=params, timeout=180)
        if response.status_code== 200:
            success= True
            # Se guardan los datos del año en response_data
            response_data= response_data +  response.json()

    


# Fecha actual con formato correcto
hoy = date.today().strftime("%Y-%m-%d")

params["start_date"]= f"2026-01-01"
params["end_date"]= hoy
    
sleep(10)
response = requests.get(url, params=params, timeout=180)
# Se guardan los datos del año en response_data
response_data= response_data +  response.json()


# guardar data en DataFrame
data = pd.DataFrame(response_data)



1996
1997
1998
1999
1999
2000
2001
2002
2003
2004
2004
2005
2005
2005
2005
2006
2006
2006
2007
2007
2007
2007
2007
2007
2007
2007
2007
2007
2008
2009
2010
2010
2011
2012
2013
2013
2013
2014
2015
2015
2015
2016
2016
2017
2017
2018
2018
2019
2020
2021
2021
2021
2022
2023
2024
2024
2025
2025
2025


## Filtrando por palabra clave

In [8]:
palabra_clave= 'spiral'
# Nos quedamos con los renglones que contienen la palabra "spiral" en la columna explanation 
# (primero transformamos la columna a minúsculas)
data_filtrada= data[data.explanation.str.lower().str.contains('spiral')]
# Convertimos la columna date en datetime (Originalmente es string)
data_filtrada.loc[:, 'date'] = pd.to_datetime(data_filtrada['date'])

# Preguntas

## 1. Para las búsquedas de tu palabra, ¿cuántos resultados obtuviste?

El número de resultados es igual al número de renglones del dataframe filtrado:

In [9]:
print( len(data_filtrada) )

1279


## 2. Para las búsquedas de tu palabra, ¿en qué rangos de fechas se introdujo el recurso?

Buscamos las fecha mínima y máxima del dataframe filtrado:

Primera fecha en la que aparece la palabra:

In [10]:
min_date= data_filtrada.date.min().strftime('%d/%m/%Y')
max_date= data_filtrada.date.max().strftime('%d/%m/%Y')

print(f"Rango: {min_date} - {max_date}") 

Rango: 26/06/1995 - 17/05/2026


## 3. Para las búsquedas de tu palabra, ¿cuáles son los "media_type" más comunes?



In [11]:
# Contamos el número de veces que se repite cada valor usando value_counts
media_count= data_filtrada.media_type.value_counts()
print("Cuenta de media_type \n")
print(media_count)

print("")

print(f"Media type con más comun: {media_count.idxmax()} ({media_count.max()} filas)" )

Cuenta de media_type 

media_type
image    1255
video      24
Name: count, dtype: int64

Media type con más comun: image (1255 filas)


## 4. Para las búsquedas de tu palabra, ¿quiénes son los autores o instituciones propietaria de los derechos (i.e. el copyright)?

In [12]:
# Series con autores de las imágenes
autores= data_filtrada.copyright
# Filtramos Nans
autores= autores.dropna()
# Lista con valores únicos de los autores
list(autores.unique())

["Anglo-Australian Telescope\nBoard\n\n Explanation:  \nLong winding spiral arms are clearly evident on this spectacular picture of\nthe spiral \ngalaxy M83.  The blue color of the \nspiral arms is caused by the relatively large fraction of young blue stars \nthere.  Dark \ndust lanes are\nmixed in with the stars and trace the spiral structure of the galaxy. This\ngalaxy contains many billions of stars, and its light took many millions of\nyears to reach us. Our own \nMilky Way Galaxy would appear similar to this if\nviewed from M83!\nThis picture is number eight on a publicly posted list of\nimages from the \nAnglo-Australian \nTelescope (AAT). \n\n\n Tomorrow's picture: Elliptical Galaxy M87 \n \n\n| Archive \n| Glossary \n| Education \n| About APOD |\n \n\nAstronomy Picture of the Day (TM) is created and copyrighted in 1995 by \nRobert \nNemiroff and \nJerry \nBonnell who are solely responsible for its content.",
 "Anglo-Australian Telescope\nBoard\n\n Explanation:  \nElliptical gal